## PENDEKATAN 2 INSET VADER WITH STRUCTURAL LEXICON ADAPTATION (ISV-SLA)


Bagian ini adalah pendekatan 2 dengan perlakuan kata fungsi 1

In [1]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os

# Konfigurasi path
DATA_PATH = '../../../datapreprocessingcopy/data_preprocessing_final.csv'
SLA_LEXICON_PATH = '../../outputs/SLA/sla_lexicon_adapted.csv' # Memuat leksikon yang sudah dibuat
OUTPUT_DIR = '../../outputs/SLA'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [2]:
# 2.2 Load Data Preprocessing Final
df = pd.read_csv(DATA_PATH)

print(f"\nData preprocessing berhasil dimuat: {len(df)} tweet")
print(f"Kolom: {df.columns.tolist()}")
df.head()


Data preprocessing berhasil dimuat: 13192 tweet
Kolom: ['no', 'timestamp', 'teks', 'teks_processed']


,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,tertib media online DPR pemerintah jangan spor...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus evaluasi lagi kebijakan bebas visa utama...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang pengaturan logis apa undang un...
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,bebas suara dapat memang jamin UU tetapi bebas...


In [3]:
# 2.3 Load Leksikon SLA yang Telah Diadaptasi
df_sla = pd.read_csv(SLA_LEXICON_PATH)
df_sla['kata'] = df_sla['kata'].astype(str).str.strip().str.lower()

# Dictionary lexicon untuk matching (SLA menggunakan kolom 'mean' sebagai skor)
sla_dict = dict(zip(df_sla['kata'], df_sla['mean']))

print(f"[INFO] Leksikon SLA berhasil dimuat: {len(sla_dict)} entri.")

[INFO] Leksikon SLA berhasil dimuat: 9071 entri.


In [8]:
# 2.4 Definisi Kategori Kata Fungsi
NEGASI_DAN_MODAL = {
    'tidak', 'bukan', 'jangan', 'belum', 'sangat', 'harus', 'wajib',
    'akan', 'sudah', 'sedang', 'telah', 'boleh', 'bisa'
}
KATA_HUBUNG_PREPOSISI = {
    'dan', 'atau', 'tetapi', 'karena', 'jika', 'di', 'ke', 'dari',
    'pada', 'untuk', 'dengan', 'oleh', 'hingga', 'sejak'
}
PRONOMINA_DEMONSTRATIVA = {
    'saya', 'aku', 'dia', 'kami', 'kamu', 'anda', 'ini', 'itu', 'yang'
}
PARTIKEL_KATA_TANYA = {
    'pun', 'sih', 'ya', 'lah', 'kah', 'apa', 'siapa', 'bagaimana'
}

ALL_FUNCTION_WORDS = (
    NEGASI_DAN_MODAL
    | KATA_HUBUNG_PREPOSISI
    | PRONOMINA_DEMONSTRATIVA
    | PARTIKEL_KATA_TANYA
)

In [9]:
# 2.5 Konfigurasi Ignore Set (Hanya yang akan dinetralkan)
IGNORE_CATEGORIES = KATA_HUBUNG_PREPOSISI | PRONOMINA_DEMONSTRATIVA | PARTIKEL_KATA_TANYA
ignore_set_final = set(df_sla[df_sla['kata'].isin(IGNORE_CATEGORIES)]['kata'])

print(f"[KONFIGURASI] Kata fungsi yang akan dinetralkan (skor=0): {len(ignore_set_final)} kata.")

[KONFIGURASI] Kata fungsi yang akan dinetralkan (skor=0): 13 kata.


In [10]:
# 2.6 Fungsi Tokenisasi
def tokenize(text):
    if not isinstance(text, str):
        return []
    return text.split()

df['tokens'] = df['teks_processed'].apply(tokenize)

print(f"[INFO] Tokenisasi selesai. Total token: {df['tokens'].str.len().sum():,}")

[INFO] Tokenisasi selesai. Total token: 235,560


In [11]:
# 2.7 Fungsi Lexicon Matching dengan Ignore Function
def match_lexicon_with_ignore(tokens, lexicon, ignore_set):
    matched_words = []
    ignored_words = []
    unmatched_words = []
    
    for token in tokens:
        token_lower = token.lower()
        if token_lower in ignore_set:
            ignored_words.append(token)
        elif token_lower in lexicon:
            matched_words.append(token)
        else:
            unmatched_words.append(token)
            
    return matched_words, ignored_words, unmatched_words

In [13]:
# 2.8 Penerapan Lexicon Matching
print("\n[PROSES] Menjalankan lexicon matching SLA dengan ignore function...")

df[['matched_words', 'ignored_words', 'unmatched_words']] = pd.DataFrame(
    df['tokens'].apply(lambda x: match_lexicon_with_ignore(x, sla_dict, ignore_set_final)).tolist(),
    index=df.index
)

print("[INFO] Lexicon matching selesai.")


[PROSES] Menjalankan lexicon matching SLA dengan ignore function...
[INFO] Lexicon matching selesai.


In [14]:
# 2.9 Perhitungan Statistik
total_words = df['tokens'].str.len().sum()
total_matched = df['matched_words'].str.len().sum()
total_ignored = df['ignored_words'].str.len().sum()
total_unmatched = df['unmatched_words'].str.len().sum()

print("\n[STATISTIK] Hasil Lexicon Matching (SLA + Ignore Function):")
print(f"Total kata             : {total_words:,}")
print(f"Matched di SLA         : {total_matched:,} ({(total_matched/total_words)*100:.2f}%)")
print(f"Ignored (dinetralkan)  : {total_ignored:,} ({(total_ignored/total_words)*100:.2f}%)")
print(f"Unmatched              : {total_unmatched:,} ({(total_unmatched/total_words)*100:.2f}%)")
print(f"Coverage Rate          : {((total_matched + total_ignored)/total_words)*100:.2f}%")


[STATISTIK] Hasil Lexicon Matching (SLA + Ignore Function):
Total kata             : 235,560
Matched di SLA         : 83,100 (35.28%)
Ignored (dinetralkan)  : 8,599 (3.65%)
Unmatched              : 143,861 (61.07%)
Coverage Rate          : 38.93%


In [16]:
# 2.9.1
# 1. Kumpulkan semua kata unmatched dari dataframe hasil matching
from collections import Counter

all_unmatched = []
for lst in df['unmatched_words']:  
    all_unmatched.extend([w.lower() for w in lst])

# 2. Hitung frekuensi dan ambil top 50
unmatched_freq = Counter(all_unmatched)
top_50_unmatched = unmatched_freq.most_common(50)

# 3. Tampilkan
print("TOP 50 KATA UNMATCHED PALING SERING MUNCUL:")
print(f"{'Kata':<20} | {'Frekuensi':<10}")
print("-" * 35)
for word, freq in top_50_unmatched:
    print(f"{word:<20} | {freq:<10}")

TOP 50 KATA UNMATCHED PALING SERING MUNCUL:
Kata                 | Frekuensi 
-----------------------------------
dpr                  | 9948      
ruu                  | 6107      
ri                   | 5492      
rakyat               | 3217      
di                   | 3158      
dan                  | 3101      
wakil                | 2451      
ini                  | 1902      
reses                | 1746      
?                    | 1613      
untuk                | 1515      
dengan               | 1434      
ke                   | 1365      
pemerintah           | 1249      
tahun                | 1127      
ketua                | 1118      
partai               | 1086      
masa                 | 944       
!                    | 932       
aspirasi             | 918       
jika                 | 765       
uu                   | 754       
akan                 | 704       
ii                   | 703       
pemilu               | 694       
oleh                 | 638       
in

In [15]:
# 2.10 Preview Hasil Matching
print("\n[PREVIEW] 5 Tweet Pertama:")
for i in range(5):
    print(f"\nTweet {i+1}: {df['teks_processed'].iloc[i][:80]}...")
    print(f"  Matched : {df['matched_words'].iloc[i]}")
    print(f"  Ignored : {df['ignored_words'].iloc[i]}")


[PREVIEW] 5 Tweet Pertama:

Tweet 1: ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !...
  Matched : ['ADIL', 'punya', 'kebijakan', 'ingat']
  Ignored : ['yang', 'yang']

Tweet 2: tertib media online DPR pemerintah jangan sporadis apalagi selektif hanya kepada...
  Matched : ['tertib', 'jangan', 'sporadis', 'selektif', 'hanya']
  Ignored : ['yang']

Tweet 3: harus evaluasi lagi kebijakan bebas visa utama untuk negara tiongkok pak ! ! bah...
  Matched : ['harus', 'lagi', 'kebijakan', 'bebas', 'bahaya', 'martabat']
  Ignored : []

Tweet 4: jangan ngambang pengaturan logis apa undang undang...
  Matched : ['jangan', 'undang', 'undang']
  Ignored : ['apa']

Tweet 5: bebas suara dapat memang jamin UU tetapi bebas sebut tidak harus bablas sehingga...
  Matched : ['bebas', 'suara', 'dapat', 'memang', 'jamin', 'bebas', 'tidak', 'harus', 'bablas', 'tabrak']
  Ignored : []


In [ ]:
# 2.11 Simpan Output
output_path = os.path.join(OUTPUT_DIR, 'sla_lexicon_matching_with_ignore.csv')
df.to_csv(output_path, index=False)

print(f"\n[OUTPUT] Data berhasil disimpan ke: {output_path}")


[OUTPUT] Data berhasil disimpan ke: ../../outputs/SLA\sla_lexicon_matching_with_ignore.csv
